# 03 - Video models on person crops (Colab A100)

Trains and predicts, in one session, the two video models of the best ensemble: V-JEPA 2 ViT-L
(`configs/colab_vjepa2.yaml`) and VideoMAE-B (`configs/colab_videomae.yaml`), all 5 subject folds, each stopped at
epoch 2 of its schedule (`train.stop_epoch`, the measured fold-mean peak). About 70 min on an A100 (40 or 80 GB); the
micro-batch and loader workers adapt to the GPU and CPUs while the effective batch stays the measured one.

Runs in the browser (Drive -> Open with Colab) or in VS Code with the Colab extension. Setup (once): copy the repo's
`src/`, `configs/`, `requirements.txt` and `secrets/kaggle.json` to Google Drive `MyDrive/DataSlayer26/`, plus
`cache/pose_cache.tgz` (YOLO11s person boxes; otherwise extracted here, ~17 min). The competition data and the frame
cache are restored from private Drive copies (`cache/data-slayer-4.zip`, `cache/frames_cache.tar`, written once by the
save cell) or downloaded / extracted on the first session. Never share that Drive folder (competition FAQ 16).

Training runs as a background queue on the VM (log: Drive `outputs/colab_train.log`), so a browser disconnect does not
stop it; the last cell only follows the log. If the VM itself dies, run all cells again: `train.resume` skips the
folds that already have `oof.csv` + `best.pt` on Drive. Outputs go to Drive `outputs/<run name>/`: per-fold
checkpoints, `oof.csv`, `cv.json`, `test_probs.csv`, plus the flip-TTA variants `*_flip.*` (validation and test both
with and without flip TTA, chosen afterwards on OOF). Every test clip is predicted on its own (FAQ 27). A failed step
stops the queue and the last cell raises. Runtime -> Change runtime type -> A100 (L4 also works, slower).

In [ ]:
import os
import shutil
import subprocess

from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/DataSlayer26"
RUNS = ["configs/colab_vjepa2.yaml", "configs/colab_videomae.yaml"]  # trained in this order, each predicted right after
for need in ["src/training/engine.py", "secrets/kaggle.json", *RUNS]:
    if not os.path.exists(f"{ROOT}/{need}"):
        raise FileNotFoundError(f"{ROOT}/{need} missing: wait for Drive desktop sync on the laptop")
%cd {ROOT}
!pip -q install timm accelerate kaggle "transformers>=4.53"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.environ["KAGGLE_CONFIG_DIR"] = f"{ROOT}/secrets"
os.environ["PYTHONPATH"] = f"{ROOT}/src"
os.environ["TQDM_MININTERVAL"] = "5"  # progress bars redraw in place every 5 s
DATA_ZIP = f"{ROOT}/cache/data-slayer-4.zip"  # private copy saved once by the save cell below (6.4 GB); never share it
if not os.path.exists("/content/data/sample_submission.csv"):
    if not os.path.exists(DATA_ZIP):
        # A cancelled download leaves a partial zip that the Kaggle CLI would skip as "up to date".
        !rm -f /content/data-slayer-4.zip
        !kaggle competitions download -c data-slayer-4 -p /content
    zip_path = DATA_ZIP if os.path.exists(DATA_ZIP) else "/content/data-slayer-4.zip"
    # Unzip to a temp dir and move in only when complete, so sample_submission.csv marks a finished copy.
    !rm -rf /content/data /content/data_tmp && unzip -q -o {zip_path} -d /content/data_tmp && mv /content/data_tmp /content/data
!ls /content/data

In [ ]:
import torch
import yaml

FRAMES_TAR = f"{ROOT}/cache/frames_cache.tar"  # written once by the save cell below (13 GB, skips the 17-min prepare)
SET = ""  # extra overrides for every run, e.g. "train.folds=[3,4]" to finish the folds after a disconnect
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
# Measured training memory per model kind: (GB per clip, fixed GB). Each GPU gets the largest micro-batch that divides
# the config batch and fits in 85% of its memory, with grad_accum making up the rest: the effective step, and with it
# the measured stop_epoch=2 recipe, is identical on L4 / A100 40 GB / A100 80 GB. A larger effective batch would not be
# faster (V-JEPA 2 already saturates the GPU) and would need a new lr / EMA / epoch search.
MEM_GB = {"vjepa2": (0.43, 2.1), "videomae": (0.6, 1.5)}
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("the configs train in bf16: pick an A100, L4 or H100 runtime (not T4)")


def run_args(config: str) -> tuple[str, str]:
    """Run name and --set overrides: loader workers = CPU count, micro-batch x grad_accum = config batch."""
    spec = yaml.safe_load(open(f"{ROOT}/{config}"))
    batch = spec["train"]["batch_size"]
    per_clip, fixed = MEM_GB[spec["model"]["kind"]]
    micro = max(d for d in range(1, batch + 1) if batch % d == 0 and fixed + d * per_clip <= 0.85 * GPU_GB)
    extra = f"data.num_workers={os.cpu_count()}"
    if micro < batch:
        extra += f" train.batch_size={micro} train.grad_accum={batch // micro}"
    return spec["name"], f"{SET} {extra}".strip()


print(f"{torch.cuda.get_device_name(0)} {GPU_GB:.0f} GB | {os.cpu_count()} CPUs")
for config in RUNS:
    spec = yaml.safe_load(open(f"{ROOT}/{config}"))
    t = spec["train"]
    print(f"{spec['name']}: folds {t['folds']} | epochs {t.get('stop_epoch') or t['epochs']} of a {t['epochs']}-epoch "
          f"schedule | batch {t['batch_size']} | hue {spec['data'].get('hue')} | SET: {run_args(config)[1]!r}")
free = shutil.disk_usage("/content/drive/MyDrive").free / 2**30
if free < 12:
    raise RuntimeError(f"Drive has {free:.1f} GB free; the new checkpoints need ~9 GB")
if not os.path.exists("/content/cache/videos.csv"):
    if os.path.exists(FRAMES_TAR):
        # Unpack to a temp dir and move in only after tar succeeds, so an interrupted restore never leaves a
        # videos.csv next to missing frames (whatever the member order in the tar).
        !rm -rf /content/cache_tmp /content/cache/frames && mkdir -p /content/cache_tmp /content/cache
        !tar -xf {FRAMES_TAR} -C /content/cache_tmp && mv /content/cache_tmp/frames /content/cache/ && mv /content/cache_tmp/videos.csv /content/cache/
    else:
        !python -m cli.prepare --config {RUNS[0]} --workers {os.cpu_count()}
!ls /content/cache/frames | wc -l

In [ ]:
POSE_TAR = f"{ROOT}/cache/pose_cache.tgz"  # YOLO11s person boxes for the crops (data.crop_features)
if not os.path.exists("/content/cache/features/yolo11s-pose_360x640_s2/test"):
    if os.path.exists(POSE_TAR):
        !mkdir -p /content/cache/features && tar -xzf {POSE_TAR} -C /content/cache/features
    else:
        !pip -q install ultralytics
        !python -m cli.extract --config configs/feat_pose.yaml --set features.backbone=yolo11s-pose paths.data_dir=/content/data paths.cache_dir=/content/cache --workers 8
!ls /content/cache/features/yolo11s-pose_360x640_s2/train | wc -l

In [ ]:
# Once per Drive: save the frame cache as one uncompressed tar (JPEGs do not compress; one big file syncs far faster
# than ~190k small ones) and the competition zip. Both are written as .part and renamed, so a cut-off upload is
# never restored; restores unpack to a temp dir first, so an interrupted restore is redone next session.
saves = []
if not os.path.exists(FRAMES_TAR) and os.path.exists("/content/cache/videos.csv"):
    du = subprocess.check_output(["du", "-sb", "/content/cache/frames"])
    saves.append((FRAMES_TAR, "frames", int(du.split()[0])))
if not os.path.exists(DATA_ZIP) and os.path.exists("/content/data-slayer-4.zip"):
    saves.append((DATA_ZIP, "zip", os.path.getsize("/content/data-slayer-4.zip")))
for dst, kind, size in saves:
    free = shutil.disk_usage("/content/drive/MyDrive").free / 2**30
    print(f"{kind} {size / 2**30:.1f} GB | Drive free {free:.1f} GB")
    if free < size / 2**30 + 10:
        print("Not enough Drive space next to the ~9 GB of new checkpoints: skipped.")
    elif kind == "frames":
        !mkdir -p {ROOT}/cache && tar -cf {dst}.part -C /content/cache frames videos.csv && mv {dst}.part {dst}
    else:
        !mkdir -p {ROOT}/cache && cp /content/data-slayer-4.zip {dst}.part && mv {dst}.part {dst}
!ls -la {ROOT}/cache

In [ ]:
import codecs
import re
import sys
import time

LOG = f"{ROOT}/outputs/colab_train.log"  # on Drive: kept across VM restarts, readable from the laptop
QUEUE = "/content/run_all.sh"


def queue_running() -> bool:
    return subprocess.run(["pgrep", "-f", QUEUE], capture_output=True).returncode == 0


if queue_running():
    print("training queue already running: following its log")
else:
    steps = []
    for config in RUNS:
        name, overrides = run_args(config)
        steps += [f"echo '=== {name}: train ==='", f"python -m cli.train --config {config} --set {overrides}",
                  f"echo '=== {name}: predict ==='", f"python -m cli.predict --config {config} --set {overrides}"]
    with open(QUEUE, "w") as f:
        f.write("set -e\n" + "\n".join(steps) + "\n")
    # Detached from the kernel, so a browser disconnect cannot stop it. After a VM restart, rerunning the notebook
    # starts the queue again and train.resume skips every fold that already has oof.csv + best.pt on Drive.
    subprocess.Popen(f"bash {QUEUE} >> '{LOG}' 2>&1; echo \"=== queue exit $? ===\" >> '{LOG}'", shell=True,
                     cwd=ROOT, env=os.environ | {"PYTHONUNBUFFERED": "1"}, start_new_session=True)
    time.sleep(10)


def follow(log: str, every: float = 10.0) -> None:
    """Streams the log (tqdm bars redraw in place) until the queue exits. Keeps this cell busy so Colab does not
    treat the VM as idle; after a reconnect just rerun this cell, it only reads the log."""
    decode = codecs.getincrementaldecoder("utf-8")(errors="replace").decode
    with open(log, "rb") as f:
        f.seek(max(0, os.path.getsize(log) - 20000))
        while True:
            chunk = f.read()
            sys.stdout.write(decode(chunk))
            sys.stdout.flush()
            if not chunk and not queue_running():
                return
            time.sleep(every)


follow(LOG)
exits = re.findall(r"=== queue exit (\d+) ===", open(LOG).read()[-5000:])
if not exits or exits[-1] != "0":
    raise RuntimeError(f"training queue failed (exit {exits[-1] if exits else '?'}): see the log above")
for config in RUNS:
    name, _ = run_args(config)
    for variant in ("cv", "cv_flip"):  # the variant with the higher 5-fold OOF AUC is the one to use on test
        print(name, variant, open(f"{ROOT}/outputs/{name}/{variant}.json").read())
!ls -la {ROOT}/outputs